In [ ]:
# Codebook:
#
# Column     Description                                             Feature Type
# ---------------------------------------------------------------------------------
# Age        Age in years                                            Numerical
# Sex        (1 = male; 0 = female)                                  Categorical
# CP         Chest pain type (0, 1, 2, 3, 4)                         Categorical
# Trestbpd   Resting blood pressure (in mm Hg on admission)          Numerical
# Chol       Serum cholesterol in mg/dl                              Numerical
# FBS        fasting blood sugar in 120 mg/dl (1 = true; 0 = false)  Categorical
# RestECG    Resting electrocardiogram results (0, 1, 2)             Categorical
# Thalach    Maximum heart rate achieved                             Numerical
# Exang      Exercise induced angina (1 = yes; 0 = no)               Categorical
# Oldpeak    ST depression induced by exercise relative to rest      Numerical
# Slope      Slope of the peak exercise ST segment                   Numerical
# CA         Number of major vessels (0-3) colored by fluoroscopy    Categorical
# Thal       3 = normal; 6 = fixed defect; 7 = reversible defect     Categorical
# Target     Diagnosis of heart disease (1 = true; 0 = false)        Target

In [ ]:
# Se define la ruta del conjunto de datos del caso.

DATA = "../data/heart_disease.csv"
OUTPUT = "../submission"
ESTIMATOR = f"{OUTPUT}/estimator.pkl"

In [ ]:
# Se define la carga de los datos que alimentan el taller.

def load_data():

    import pandas as pd

    dataset = pd.read_csv(DATA)
    y = dataset.pop("target")
    x = dataset.copy()
    x["thal"] = x["thal"].map(
        lambda x: "normal" if x not in ["fixed", "fixed", "reversible"] else x
    )

    return x, y


x, y = load_data()
x

In [ ]:
# Se define la separación entre entrenamiento y prueba.

def make_train_test_split(x, y):

    from sklearn.model_selection import train_test_split

    (x_train, x_test, y_train, y_test) = train_test_split(
        x,
        y,
        test_size=0.10,
        random_state=0,
    )
    return x_train, x_test, y_train, y_test

In [ ]:
# Se define el pipeline para conservar las transformaciones junto al modelo.

def make_pipeline(estimator):

    from sklearn.compose import ColumnTransformer
    from sklearn.feature_selection import SelectKBest, f_classif
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder

    transformer = ColumnTransformer(
        transformers=[
            ("ohe", OneHotEncoder(dtype="int"), ["thal"]),
        ],
        remainder="passthrough",
    )

    selectkbest = SelectKBest(score_func=f_classif)

    pipeline = Pipeline(
        steps=[
            ("tranformer", transformer),
            ("selectkbest", selectkbest),
            ("estimator", estimator),
        ],
        verbose=False,
    )

    return pipeline

In [ ]:
# Se define la búsqueda de parámetros mediante validación cruzada.

def make_grid_search(estimator, param_grid, cv=5):

    from sklearn.model_selection import GridSearchCV

    grid_search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=cv,
        scoring="balanced_accuracy",
    )

    return grid_search

In [ ]:
# Se define el almacenamiento del estimador para reutilizarlo posteriormente.

def save_estimator(estimator, x, y):

    import pickle

    from sklearn.metrics import balanced_accuracy_score

    best_estimator = load_estimator()

    if best_estimator is not None:

        saved_accuracy = balanced_accuracy_score(
            y_true=y, y_pred=best_estimator.predict(x)
        )

        current_accuracy = balanced_accuracy_score(
            y_true=y, y_pred=estimator.predict(x)
        )

        if saved_accuracy < current_accuracy:
            estimator = best_estimator

    with open(ESTIMATOR, "wb") as file:
        pickle.dump(estimator, file)

In [ ]:
# Se define la carga del estimador almacenado.

def load_estimator():

    import os
    import pickle

    if not os.path.exists(ESTIMATOR):
        return None

    with open(ESTIMATOR, "rb") as file:
        estimator = pickle.load(file)

    return estimator

In [ ]:
# Se define el entrenamiento para repetir el flujo sin omitir pasos.

def train_estimator(estimator):

    from sklearn.linear_model import LogisticRegression

    data, target = load_data()

    x_train, x_test, y_train, y_test = make_train_test_split(
        x=data,
        y=target,
    )

    estimator.fit(x_train, y_train)

    save_estimator(estimator, x_test, y_test)

In [ ]:
# Se define el entrenamiento de la regresión logística.

def train_logistic_regression():

    from sklearn.linear_model import LogisticRegression

    pipeline = make_pipeline(
        estimator=LogisticRegression(max_iter=10000, solver="saga"),
    )

    param_grid = {
        "selectkbest__k": range(1, 11),
        "estimator__penalty": ["l1", "l2"],
        "estimator__C": [0.001, 0.01, 0.1, 1, 10, 100],
    }

    estimator = make_grid_search(
        estimator=pipeline,
        param_grid=param_grid,
        cv=5,
    )

    train_estimator(estimator)


train_logistic_regression()

In [ ]:
# Se definen las métricas para comparar los modelos.

def eval_metrics(
    y_train_true,
    y_test_true,
    y_train_pred,
    y_test_pred,
):

    from sklearn.metrics import accuracy_score, balanced_accuracy_score

    accuracy_train = round(accuracy_score(y_train_true, y_train_pred), 4)
    accuracy_test = round(accuracy_score(y_test_true, y_test_pred), 4)
    balanced_accuracy_train = round(
        balanced_accuracy_score(y_train_true, y_train_pred), 4
    )
    balanced_accuracy_test = round(balanced_accuracy_score(y_test_true, y_test_pred), 4)

    return (
        accuracy_train,
        accuracy_test,
        balanced_accuracy_train,
        balanced_accuracy_test,
    )

In [ ]:
# Se define el reporte de métricas para discutir la comparación.

def report(
    estimator,
    accuracy_train,
    accuracy_test,
    balanced_accuracy_train,
    balanced_accuracy_test,
):

    print(estimator, ":", sep="")
    print("-" * 80)
    print(f"Balanced Accuracy: {balanced_accuracy_test} ({balanced_accuracy_train})")
    print(f"         Accuracy: {accuracy_test} ({accuracy_train})")

In [ ]:
# Se define la verificación del estimador que quedó almacenado.

def check_estimator():

    data, target = load_data()

    x_train, x_test, y_train_true, y_test_true = make_train_test_split(
        x=data,
        y=target,
    )

    estimator = load_estimator()

    y_train_pred = estimator.predict(x_train)
    y_test_pred = estimator.predict(x_test)

    (
        accuracy_train,
        accuracy_test,
        balanced_accuracy_train,
        balanced_accuracy_test,
    ) = eval_metrics(
        y_train_true,
        y_test_true,
        y_train_pred,
        y_test_pred,
    )

    report(
        estimator.best_estimator_,
        accuracy_train,
        accuracy_test,
        balanced_accuracy_train,
        balanced_accuracy_test,
    )


check_estimator()

In [ ]:
# Se define el entrenamiento del clasificador neuronal para compararlo.

def train_mlp_classifier():

    from sklearn.neural_network import MLPClassifier

    pipeline = make_pipeline(
        estimator=MLPClassifier(max_iter=1000),
    )

    param_grid = {
        "selectkbest__k": range(1, 5),
        "estimator__hidden_layer_sizes": [1, 2, 3],
        "estimator__learning_rate_init": [0.0001, 0.001, 0.01, 0.1, 1.0],
    }

    estimator = make_grid_search(
        estimator=pipeline,
        param_grid=param_grid,
        cv=5,
    )

    train_estimator(estimator)


# train_mlp_classifier()
# check_estimator()